In [ ]:
import json
import pandas as pd
from pathlib import Path
import re
import numpy as np

1. Load the Restaurant
2. Basic Cleaning

In [ ]:
import pyarrow as pa
import pyarrow.json as paj
import pyarrow.parquet as pq

In [ ]:
input = Path("../data/processed/restaurants.parquet")
restaurant_df = pd.read_parquet(input)
restaurant_df.shape

In [ ]:
# Remove NA Lat/Long and Invalid
restaurant_df = restaurant_df[
    restaurant_df["latitude"].notna() &
    restaurant_df["longitude"].notna()
]
# Lat between (-90, 90), Long(-180, 180)
restaurant_df = restaurant_df[
    restaurant_df["latitude"].between(-90, 90) &
    restaurant_df["longitude"].between(-180, 180)
]

In [ ]:
restaurant_df.shape

In [ ]:
open_restaurants = restaurant_df
open_restaurants.head()

In [ ]:
cols_to_keep = ["business_id", "name","city", "state", "latitude",
                "longitude", "stars", "review_count", "categories",
                "attributes", "hours", "price_tier", "tz"]

open_restaurants = open_restaurants[cols_to_keep]
open_restaurants = open_restaurants.reset_index(drop=True)

In [ ]:
import ast
def _strip_unicode_prefixes(s):
    return re.sub(r"""(^|[\s{,\[])u(['"])""", r"\1\2", s)

def _fix_weird_quotes(s: str) -> str:
    """ Fix common corruptions and double-single-quote endings."""
    s = re.sub(r"(')([^']*)''", r"\1\2'", s)
    s = re.sub(r'(")([^"]*)""', r'\1\2"', s)
    return s

def parse_literal(x, want_type=None):
    """ Parse dict/list stored as"""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {} if want_type is dict else ([] if want_type is list else None)

    if want_type is not None and isinstance(x, want_type):
        return x
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return {} if want_type is dict else ([] if want_type is list else None)

    s = x.strip()
    if not s or s.lower() in {"none", "null", "nan"}:
        return {} if want_type is dict else ([] if want_type is list else None)

    s = _strip_unicode_prefixes(s)
    s = _fix_weird_quotes(s)

    if want_type is list and s.startswith("[") and s.endswith("]"):
        inner = s[1:-1].strip()
        if inner and ("'" not in inner and '"' not in inner):
            parts = [p.strip() for p in inner.split(",")]
            s = "[" + ", ".join(repr(p) for p in parts if p) + "]"

    try:
        obj = ast.literal_eval(s)
    except Exception:
        return {} if want_type is dict else ([] if want_type is list else None)

    if want_type is not None:
        return obj if isinstance(obj, want_type) else ({} if want_type is dict else [])
    return obj

In [ ]:
# Parse (Remove) Restaurant, Food, (repetitive) tokens

def parse_categories(cat_val) -> list[str]:
    if cat_val is None or (isinstance(cat_val, float) and pd.isna(cat_val)):
        return []

    if isinstance(cat_val, np.ndarray):
        if cat_val.ndim == 0:
            cat_val = cat_val.item()
        else:
            cat_val = cat_val.tolist()
    if isinstance(cat_val, list):
        raw = cat_val

    # Common Words to Drop
    drop = {"restaurants", "food"}
    cleaned = []
    for t in raw:
        if not isinstance(t, str):
            continue
        t = t.strip().lower()
        if not t or t in drop:
            continue
        t = t.replace("&", "and")
        t = re.sub(r"\s+", " ", t)
        t = t.strip(" -_/[]()\"'")
        if t and t not in drop:
            cleaned.append(t)

    out, seen = [], set()
    for c in cleaned:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

open_restaurants = open_restaurants.copy()
open_restaurants["category_tokens"] =(
    open_restaurants["categories"].apply(parse_categories))

# Filter top 5 Metros

In [ ]:
# Keep Restaurants with Min_Review > Threshold, Compute metros from training population
Min_Reviews = 20
train_pool = open_restaurants[open_restaurants["review_count"] >= Min_Reviews].copy()

In [ ]:
metro_counts = (train_pool.groupby(["city", "state"]).
                size().sort_values(ascending=False))
top_n = 5
top_metros = list(metro_counts.head(top_n).index)

In [ ]:
metro_mask = open_restaurants.set_index(["city", "state"]).index.isin(top_metros)
open_restaurants = open_restaurants[metro_mask].reset_index(drop=True)

# Rebuild train_restaurants after metro filter
train_restaurants = open_restaurants[open_restaurants["review_count"] >= Min_Reviews].copy()

## 3. Build the Vocabulary

In [ ]:
from collections import Counter
train_restaurants = train_restaurants.copy()
train_restaurants["category_tokens"] = train_restaurants["categories"].apply(parse_categories)

# Between 300-1000
K = 500
min_df = 30
max_df_ratio = 0.35

stop_cats = {
    "food", "restaurants", "bars", "nightlife",
    "local flavor", "event planning and services", "shopping"
}

df_counter = Counter()
for toks in train_restaurants["category_tokens"]:
    uniq = set(toks) - stop_cats
    df_counter.update(uniq)

n_docs = len(train_restaurants)
max_df = int(max_df_ratio * n_docs)

candidates = [(tok, df) for tok, df in df_counter.items() if min_df <= df <= max_df]
candidates.sort(key=lambda x: (-x[1], x[0]))
top_tokens = [tok for tok, _ in candidates[:K]]

category2idx = {"<PAD>": 0, "<UNK>": 1}
category2idx.update({tok: i for i, tok in enumerate(top_tokens, start=2)})  # <-- start=2

idx2category = {i: tok for tok, i in category2idx.items()}

print("Top DF tokens:", df_counter.most_common(10))

In [ ]:
# Map OOV to UNK
def tokens_to_indices(tokens, vocab, unk_id=1):
    if not tokens:
        return [unk_id]
    out, seen = [], set()
    for t in tokens:
        i = vocab.get(t, unk_id)
        if i not in seen:
            out.append(i)
            seen.add(i)
    return out if out else [unk_id]

In [ ]:
open_restaurants["category_ids"] = open_restaurants["category_tokens"].apply(
    lambda toks:tokens_to_indices(toks, category2idx))

train_restaurants["category_ids"] = train_restaurants["category_tokens"].apply(
    lambda toks: tokens_to_indices(toks, category2idx))

In [ ]:
open_restaurants.head()

In [ ]:
cuisines_and_food_types = {'thai', 'japanese', 'chinese', 'mexican', 'italian', 'indian',
    'korean', 'vietnamese', 'french', 'greek', 'mediterranean',
    'american (traditional)', 'american (new)', 'southern', 'cajun/creole',
    'caribbean', 'latin american',
     'pakistani', 'cuban',
    'asian fusion', 'tex-mex', 'soul food','pizza', 'ramen', 'sandwiches', 'seafood', 'steakhouses',
    'sushi bars', 'noodles', 'barbeque', 'breakfast & brunch',
    'coffee & tea', 'bakeries', 'desserts', 'ice cream & frozen yogurt',
    'juice bars & smoothies', 'fast food', 'diners', 'buffets',
    'food trucks', 'comfort food', 'chicken wings', 'tacos',
    'hot dogs', 'salad', 'soup'}

In [ ]:
cuisine_vocab = sorted(cuisines_and_food_types)
cuisine2idx = {c: i for i, c in enumerate(cuisine_vocab)}

def cuisine_vector(tokens, vocab=cuisine_vocab):
    s = set(tokens)
    return np.array([c in s for c in vocab], dtype=bool)

In [ ]:
for df in [train_restaurants, open_restaurants]:
    df["cuisine_vec"] = df["category_tokens"].apply(lambda x: cuisine_vector(x).astype(np.int8))

In [ ]:
open_restaurants["cuisine_vec"].head()

## 4. Parsing the Price Tiers

In [ ]:
"""def extract_price(attributes):

    if not isinstance(attributes, dict):
        return 0
    val = attributes.get("RestaurantsPriceRange2")
    if val is None:
        return 0
    try:
        price = int(val)
        if 1 <= price <= 4:
            return price
    except:
        pass
    return 0"""

In [ ]:
"""open_restaurants["price_tier"] = open_restaurants["attributes"].apply(extract_price)
train_restaurants["price_tier"] = train_restaurants["attributes"].apply(extract_price)"""

In [ ]:
"""# Sanity Check
open_restaurants["price_tier"].value_counts(dropna=False)"""

In [ ]:
"""# Price Embedding
import torch
import torch.nn as nn

d_price = 8"""

## 5. Parsing the Attribute Data

In [ ]:
def normalize_str(x):
    """ Normalize strings"""
    if not isinstance(x, str):
        return x
    s = x.strip()

    # Remove python unicode prefix u
    s = re.sub(r"""^u(['"])""", r"\1", s)

    # Strip repeated ""
    while len(s) >= 2 and ((s[0] == s[-1] == "'")
                           or (s[0] == s[-1] == '"')):
        s = s[1:-1].strip()
    s = s.strip("'").strip('"').strip()

    return s

def normalize_categorical(x):
    """Return normalized lowercase categorical string or None."""
    if x is None:
        return None
    if not isinstance(x, str):
        return None
    s = normalize_str(x).lower()
    if s in {"none", "null", ""}:
        return None
    return s

In [ ]:
# Function to convert True/False to 1, 0
true_set = {"true", "1", "yes", "y", "t"}
false_set = {"false", "0", "no", "n", "f"}

def to_bool(x):
    """Convert Yelp attribute values to bool or None."""
    if x is None:
        return None
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)) and x in (0, 1):
        return bool(x)

    if isinstance(x, str):
        s = normalize_str(x).lower()
        if s in true_set:
            return True
        if s in false_set:
            return False
        if s in {"none", "null", ""}:
            return None

    return None

In [ ]:
# Parse Attributes into Dict
import ast
def parse_dict(x):
    if isinstance(x, dict):
        return x
    if x is None:
        return {}
    if not isinstance(x, str):
        return {}

    txt = x.strip()
    if not txt or txt.lower() in {"none", "null"}:
        return {}

    # Remove unicode prefixes
    txt = re.sub(r"""u(['"])""", r"\1", txt)

    try:
        obj = ast.literal_eval(txt)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}

In [ ]:
# Binary Features
binary_keys = [
    "RestaurantsReservations",
    "OutdoorSeating",
    "HasTV",
    "GoodForKids",
    "BusinessAcceptsCreditCards"]

# Categorical One Hots
wifi_lvls = ["free", "paid", "no"]
alc_lvls = ["none", "beer_and_wine", "full_bar"]
noise_lvls = ["quiet", "average", "loud", "very_loud"]
attire_lvls = ["casual", "dressy", "formal"]

meal_keys = ["breakfast", "brunch", "lunch", "dinner", "latenight"]
ambience_keys = ["casual", "romantic", "touristy", "hipster", "upscale"]

# parking_keys = ["garage", "street", "validated", "lot", "valet"]

In [ ]:
def attributes_to_features(raw_attributes):
    attrs = parse_dict(raw_attributes)
    feats = {}

    # Binary Flags
    for k in binary_keys:
        feats[f"attr_{k}"] = int(to_bool(attrs.get(k)) is True)

    # Wifi (OHE)
    wifi = normalize_categorical(attrs.get("WiFi"))
    for level in wifi_lvls:
        feats[f"wifi_{level}"] = int(wifi == level)

    feats["wifi_unknown"] = int(wifi is None
                                or wifi not in wifi_lvls)

    # Alcohol (OHE)
    alcohol = normalize_categorical(attrs.get("Alcohol"))
    for level in alc_lvls:
        feats[f"alcohol_{level}"] = int(alcohol == level)
    feats["alcohol_unknown"] = int(alcohol is None
                                   or alcohol not in alc_lvls)

    # Noise (OHE)
    noise = normalize_categorical(attrs.get("NoiseLevel"))
    for level in noise_lvls:
        feats[f"noise_{level}"] = int(noise == level)
    feats["noise_unknown"] = int(noise is None
                                 or noise not in noise_lvls)

    # Attire (OHE)
    attire = normalize_categorical(attrs.get("RestaurantsAttire"))
    for level in attire_lvls:
        feats[f"attire_{level}"] = int(attire == level)
    feats["attire_unknown"] = int(attire is None
                                  or attire not in attire_lvls)

    # Nested Dictionary
    good_for_meal = parse_dict(attrs.get("GoodForMeal"))
    for mk in meal_keys:
        feats[f"meal_{mk}"] = int(to_bool(good_for_meal.get(mk)) is True)

    ambience = parse_dict(attrs.get("Ambience"))
    for ak in ambience_keys:
        feats[f"ambience_{ak}"] = int(to_bool(ambience.get(ak)) is True)

    """parking = parse_dict(attrs.get("BusinessParking"))
    for pk in parking_keys:
        feats[f"parking_{pk}"] = int(to_bool(parking.get(pk)) is True)"""

    return feats

In [ ]:
feat_df = open_restaurants["attributes"].apply(attributes_to_features).apply(pd.Series)
open_restaurants = pd.concat([open_restaurants.reset_index(drop=True),
                              feat_df.reset_index(drop=True)], axis=1)

In [ ]:
open_restaurants.head()

## 6. Parsing Time Data

In [ ]:
days = ["Monday","Tuesday","Wednesday","Thursday",
        "Friday","Saturday","Sunday"]
day_idx = {d:i for i,d in enumerate(days)}
time_re = re.compile(r"^\s*(\d{1,2})\s*:\s*(\d{1,2})\s*$")

In [ ]:
def hhmm_to_min(t, allow_24 = True):
    if not isinstance(t, str):
        return None
    m = time_re.match(t)
    if not m:
        return None

    hh = int(m.group(1))
    mm = int(m.group(2))

    if allow_24 and hh == 24 and mm == 0:
        return 1440

    if not (0 <= hh <= 23 and 0 <= mm <= 59):
        return None
    return hh * 60 + mm

In [ ]:
def hours_to_dict(hours_val):
    d = parse_literal(hours_val, want_type=dict)
    return d if d else None

In [ ]:
def merge_intervals(intervals):
    """Merge overlapping or adjacent time intervals"""
    if not intervals:
        return intervals
    merged = [intervals[0]]

    for i, j in intervals[1:]:
        ps, pe = merged[-1]
        if i <= pe:
            merged[-1] = (ps, max(pe, j))
        else:
            merged.append((i, j))
    return merged

In [ ]:
def parse_hours_dict(hours, handle_0000 = True):
    schedule = [[] for _ in range(7)]

    hours_dict = hours_to_dict(hours)
    if not isinstance(hours_dict, dict) or not hours_dict:
        return schedule, False
    any_parsed = False

    for day, span in hours_dict.items():
        if day not in day_idx or not isinstance(span, str):
            continue

        parts = [p.strip() for p in span.split(",") if p.strip()]
        for p in parts:
            if "-" not in p:
                continue

            a, b = [x.strip() for x in p.split("-", 1)]
            start = hhmm_to_min(a, allow_24=True)
            end   = hhmm_to_min(b, allow_24=True)
            if start is None or end is None:
                continue


            if handle_0000 and start == 0 and end == 0:
                continue

            di = day_idx[day]

            if start < end:
                schedule[di].append((start, end))
                any_parsed = True
            elif start > end:

                schedule[di].append((start, 1440))
                schedule[(di + 1) % 7].append((0, end))
                any_parsed = True
            # start == end
            else:
                continue

    for di in range(7):
        schedule[di].sort()
        schedule[di] = merge_intervals(schedule[di])

    return schedule, any_parsed

In [ ]:
# Function to Check if Open Now
def is_open_now(schedule, weekday_idx: int, minute_of_day: int) -> bool:
    """Return True if minute_of_day falls into any interval for that weekday."""
    for start, end in schedule[weekday_idx]:
        if start <= minute_of_day < end:
            return True
    return False

In [ ]:
def interval_overlaps(a_start, a_end, b_start, b_end):
    return (a_start < b_end) and (b_start < a_end)

def hours_features(schedule, known: bool):
    if not known:
        return {"hours_known": 0,"open_late": 0,
                "open_for_lunch": 0, "open_days_count": 0}

    open_days = 0
    open_late = 0
    open_for_lunch = 0

    for day_intervals in schedule:
        if day_intervals:
            open_days += 1

        for start, end in day_intervals:
            # After 23:00
            if end > 23 * 60:
                open_late = 1
            # Before 11:00
            if interval_overlaps(start, end, 11 * 60, 14 * 60):
                open_for_lunch = 1

    return {"hours_known": 1, "open_late": open_late,
        "open_for_lunch": open_for_lunch, "open_days_count": open_days}

In [ ]:
def compute_hours_struct_and_feats(hours):
    schedule, known = parse_hours_dict(hours)
    feats = hours_features(schedule, known)
    return schedule, known, feats

In [ ]:
open_restaurants = open_restaurants.reset_index(drop=True).copy()

tmp = open_restaurants["hours"].apply(compute_hours_struct_and_feats)

open_restaurants["weekly_schedule"] = tmp.map(lambda x: x[0])
open_restaurants["hours_known"] = tmp.map(lambda x: int(x[1]))

feat_df = tmp.map(lambda x: x[2]).apply(pd.Series)
feat_df = feat_df.drop(columns=["hours_known"], errors="ignore")

open_restaurants = open_restaurants.join(feat_df)

In [ ]:
attr_cols = (
    [c for c in open_restaurants.columns if c.startswith("attr_")] +
    [c for c in open_restaurants.columns if c.startswith("wifi_")] +
    [c for c in open_restaurants.columns if c.startswith("alcohol_")] +
    [c for c in open_restaurants.columns if c.startswith("noise_")] +
    [c for c in open_restaurants.columns if c.startswith("attire_")] +
    [c for c in open_restaurants.columns if c.startswith("meal_")] +
    [c for c in open_restaurants.columns if c.startswith("ambience_")] +
    ["hours_known", "open_late", "open_for_lunch", "open_days_count"])

In [ ]:
def stable_unique(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

attr_cols = stable_unique(attr_cols)

## 7. Geographic Region Clustering

In [ ]:
import geohash2

In [ ]:
# Neighborhood Precision
def latlon_to_region(lat, lon, precision):
    try:
        return geohash2.encode(lat, lon, precision=precision)
    except:
        return None

In [ ]:
precisions = 4, 6
for p in precisions:
    open_restaurants[f"geo{p}"] = open_restaurants.apply(
        lambda r: latlon_to_region(r["latitude"], r["longitude"], precision=p),
        axis=1)
    train_restaurants[f"geo{p}"] = train_restaurants.apply(
    lambda r: latlon_to_region(r["latitude"], r["longitude"], precision=p),
    axis=1
)

# build vocab per precision for train only
geo2idx = {}
for p in precisions:
    tokens_train = train_restaurants[f"geo{p}"].dropna().tolist()
    uniq = pd.Series(tokens_train).value_counts().index.tolist()
    geo2idx[p] = {"<UNK>": 0, **{t:i+1 for i,t in enumerate(uniq)}}
    open_restaurants[f"geo{p}_id"] = open_restaurants[f"geo{p}"].map(lambda t: geo2idx[p].get(t, 0)).astype("int64")
    train_restaurants[f"geo{p}_id"] = train_restaurants[f"geo{p}"].map(lambda t: geo2idx[p].get(t, 0)).astype("int64")

In [ ]:
# Embedding Dimension 16? GeoHash7
d_geo = {4: 8, 6: 16}


In [ ]:
import numpy as np
# Fill NaNs w. 0
open_restaurants[attr_cols] = open_restaurants[attr_cols].fillna(0).astype("float32")
attr_matrix = open_restaurants[attr_cols].to_numpy(dtype=np.float32)

In [ ]:
open_restaurants["attr_vec"] = list(attr_matrix)

# Attribute Dimension
attr_dim = attr_matrix.shape[1]
print("attr_dim =", attr_dim)

In [ ]:
open_restaurants.head()

In [ ]:
restaurant_items = open_restaurants.copy()
restaurant_items = restaurant_items.rename(columns={"geo6_id": "region_id"})

keep_cols = [
    "business_id",
    "category_tokens",
    "category_ids",
    "cuisine_vec",
    "price_tier",
    "attr_vec",
    "geo4_id",
    "region_id",
    "hours_known",
    "open_late",
    "open_for_lunch",
    "open_days_count",
    "latitude",
    "longitude",
    "weekly_schedule",
    "city",
    "state"]

restaurant_items = restaurant_items[keep_cols].copy()

In [ ]:
restaurant_items.head()

In [ ]:
out_dir = Path("../data/processed/Restaurant Artifacts")

out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import json

# Category vocab
with open(out_dir / "category2idx.json", "w") as f:
    json.dump(category2idx, f)

with open(out_dir / "idx2category.json", "w") as f:
    json.dump({str(k): v for k, v in idx2category.items()}, f)

# Region vocab
with open(out_dir / "region2idx_geo6.json", "w") as f:
    json.dump(geo2idx[6], f)

# Attribute schema (attr_vec)
attr_schema = {"attr_dim": int(attr_dim), "attr_cols": attr_cols}
with open(out_dir / "attr_schema.json", "w") as f:
    json.dump(attr_schema, f)

# Save a small metadata file (helps later)
meta = {
    "top_metros_n": 5,
    "min_reviews_train": int(Min_Reviews),
    "category_vocab_topK": int(K),
    "cat_vocab_size": int(max(category2idx.values()) + 1),
    "region_vocab_size": int(max(geo2idx[6].values()) + 1),
}
with open(out_dir / "preprocess_meta.json", "w") as f:
    json.dump(meta, f, indent=2)


In [ ]:
restaurant_items.to_parquet(out_dir / "restaurant_items.parquet", index=False)

In [ ]:
df = pd.read_parquet("data/processed/Restaurant Artifacts/restaurant_items.parquet")
df.head()